# Coursera Course Dataset Analysis

## Phase 2: Data Cleaning

This notebook analyzes the Coursera Course Dataset from Kaggle. In this phase, the goal is to make the raw dataset reliable before we begin exploratory data analysis.

Data cleaning means checking whether the columns, values, and data types are suitable for analysis. If the data has an old index column, text values that should be numbers, duplicate rows, or inconsistent categories, we fix or document those issues before answering business questions.

## Dataset Description

- Source: Kaggle Coursera Course Dataset
- Raw file used in this project: `../data/coursea_data.csv`
- Each row represents one Coursera course or course-related offering.
- Main columns include course title, course organization, certificate type, rating, difficulty level, and students enrolled.

We will first load the data and inspect its basic structure.

In [1]:
from pathlib import Path

import pandas as pd

DATA_PATH = Path("../data/coursea_data.csv")
CLEANED_OUTPUT_PATH = Path("../outputs/coursera_courses_cleaned.csv")

raw_courses = pd.read_csv(DATA_PATH)

raw_courses.head()

,Unnamed: 0,course_title,course_organization,course_Certificate_type,course_rating,course_difficulty,course_students_enrolled
0,134,(ISC)² Systems Security Certified Practitioner...,(ISC)²,SPECIALIZATION,4.7,Beginner,5.3k
1,743,A Crash Course in Causality: Inferring Causal...,University of Pennsylvania,COURSE,4.7,Intermediate,17k
2,874,A Crash Course in Data Science,Johns Hopkins University,COURSE,4.5,Mixed,130k
3,413,A Law Student's Toolkit,Yale University,COURSE,4.7,Mixed,91k
4,635,A Life of Happiness and Fulfillment,Indian School of Business,COURSE,4.8,Mixed,320k


In [2]:
raw_courses.shape

(891, 7)

In [3]:
raw_courses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                891 non-null    int64  
 1   course_title              891 non-null    object 
 2   course_organization       891 non-null    object 
 3   course_Certificate_type   891 non-null    object 
 4   course_rating             891 non-null    float64
 5   course_difficulty         891 non-null    object 
 6   course_students_enrolled  891 non-null    object 
dtypes: float64(1), int64(1), object(5)
memory usage: 48.9+ KB


## Cleaning Step 1: Review Column Names

The raw data has one unnamed column. This looks like an old row index from the original export, not a real course feature. We will remove it because it does not describe the course itself.

We will also rename the remaining columns into a consistent style called `snake_case`. That means lowercase words separated by underscores, such as `course_title`.

In [4]:
raw_courses.columns.tolist()

['Unnamed: 0',
 'course_title',
 'course_organization',
 'course_Certificate_type',
 'course_rating',
 'course_difficulty',
 'course_students_enrolled']

In [5]:
courses = raw_courses.copy()

courses = courses.drop(columns=[courses.columns[0]])

courses = courses.rename(
    columns={
        "course_title": "course_title",
        "course_organization": "organization",
        "course_Certificate_type": "certificate_type",
        "course_rating": "rating",
        "course_difficulty": "difficulty",
        "course_students_enrolled": "students_enrolled_raw",
    }
)

courses.head()

,course_title,organization,certificate_type,rating,difficulty,students_enrolled_raw
0,(ISC)² Systems Security Certified Practitioner...,(ISC)²,SPECIALIZATION,4.7,Beginner,5.3k
1,A Crash Course in Causality: Inferring Causal...,University of Pennsylvania,COURSE,4.7,Intermediate,17k
2,A Crash Course in Data Science,Johns Hopkins University,COURSE,4.5,Mixed,130k
3,A Law Student's Toolkit,Yale University,COURSE,4.7,Mixed,91k
4,A Life of Happiness and Fulfillment,Indian School of Business,COURSE,4.8,Mixed,320k


## Cleaning Step 2: Check Missing Values

Missing values are empty cells. They matter because charts and calculations can become misleading if many values are missing.

In [6]:
missing_values = courses.isna().sum().to_frame(name="missing_count")
missing_values["missing_percent"] = (
    missing_values["missing_count"] / len(courses) * 100
).round(2)

missing_values

,missing_count,missing_percent
course_title,0,0.0
organization,0,0.0
certificate_type,0,0.0
rating,0,0.0
difficulty,0,0.0
students_enrolled_raw,0,0.0


## Cleaning Step 3: Check Duplicates

A duplicate row means the exact same record appears more than once. A duplicate course title means the title appears more than once, even if another column is different.

We will check both because they answer different questions.

In [7]:
duplicate_row_count = courses.duplicated().sum()
duplicate_title_count = courses["course_title"].duplicated().sum()

duplicate_row_count, duplicate_title_count

(np.int64(0), np.int64(3))

In [8]:
duplicate_titles = courses.loc[
    courses["course_title"].duplicated(keep=False)
].sort_values("course_title")

duplicate_titles

,course_title,organization,certificate_type,rating,difficulty,students_enrolled_raw
224,Developing Your Musicianship,Berklee College of Music,COURSE,4.8,Mixed,41k
225,Developing Your Musicianship,Berklee College of Music,SPECIALIZATION,4.8,Beginner,54k
563,Machine Learning,University of Washington,SPECIALIZATION,4.6,Intermediate,290k
564,Machine Learning,Stanford University,COURSE,4.9,Mixed,3.2m
582,Marketing Digital,Universidade de São Paulo,COURSE,4.8,Beginner,81k
583,Marketing Digital,Universidad Austral,SPECIALIZATION,4.7,Beginner,39k


## Cleaning Step 4: Convert Enrollment Counts

The `students_enrolled_raw` column contains values like `5.3k`, `130k`, and `2.5m`. Pandas reads these as text, but for analysis we need numeric values.

The function below converts:

- `k` into thousands
- `m` into millions

For example, `5.3k` becomes `5300`.

In [9]:
def parse_enrollment(value):
    """Convert Coursera enrollment text into a whole number."""
    text_value = str(value).strip().lower().replace(",", "")

    if text_value.endswith("k"):
        return int(float(text_value[:-1]) * 1_000)

    if text_value.endswith("m"):
        return int(float(text_value[:-1]) * 1_000_000)

    return int(float(text_value))


courses["students_enrolled"] = courses["students_enrolled_raw"].apply(
    parse_enrollment
)

courses[["students_enrolled_raw", "students_enrolled"]].head(10)

,students_enrolled_raw,students_enrolled
0,5.3k,5300
1,17k,17000
2,130k,130000
3,91k,91000
4,320k,320000
5,39k,39000
6,350k,350000
7,2.4k,2400
8,61k,61000
9,12k,12000


## Cleaning Step 5: Check Ratings

Ratings should be numeric. Coursera ratings usually use a 0 to 5 scale, so we will convert ratings to numbers and check the smallest and largest values.

In [10]:
courses["rating"] = pd.to_numeric(courses["rating"], errors="coerce")

courses["rating"].describe()

count    891.000000
mean       4.677329
std        0.162225
min        3.300000
25%        4.600000
50%        4.700000
75%        4.800000
max        5.000000
Name: rating, dtype: float64

In [11]:
invalid_ratings = courses.loc[
    courses["rating"].isna() | ~courses["rating"].between(0, 5)
]

invalid_ratings

,course_title,organization,certificate_type,rating,difficulty,students_enrolled_raw,students_enrolled


## Cleaning Step 6: Check Category Values

Category columns should have a small set of clear values. We will check certificate type and difficulty to make sure there are no spelling problems or unexpected labels.

In [12]:
courses["certificate_type"].value_counts()

certificate_type
COURSE                      582
SPECIALIZATION              297
PROFESSIONAL CERTIFICATE     12
Name: count, dtype: int64

In [13]:
courses["difficulty"].value_counts()

difficulty
Beginner        487
Intermediate    198
Mixed           187
Advanced         19
Name: count, dtype: int64

## Cleaning Step 7: Save the Cleaned Dataset

Now we save a cleaned CSV. This makes the next phase easier because exploratory data analysis can start from the cleaned version instead of repeating all cleaning steps.

In [14]:
cleaned_columns = [
    "course_title",
    "organization",
    "certificate_type",
    "rating",
    "difficulty",
    "students_enrolled_raw",
    "students_enrolled",
]

courses_cleaned = courses.loc[:, cleaned_columns].copy()

CLEANED_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
courses_cleaned.to_csv(CLEANED_OUTPUT_PATH, index=False)

courses_cleaned.head()

,course_title,organization,certificate_type,rating,difficulty,students_enrolled_raw,students_enrolled
0,(ISC)² Systems Security Certified Practitioner...,(ISC)²,SPECIALIZATION,4.7,Beginner,5.3k,5300
1,A Crash Course in Causality: Inferring Causal...,University of Pennsylvania,COURSE,4.7,Intermediate,17k,17000
2,A Crash Course in Data Science,Johns Hopkins University,COURSE,4.5,Mixed,130k,130000
3,A Law Student's Toolkit,Yale University,COURSE,4.7,Mixed,91k,91000
4,A Life of Happiness and Fulfillment,Indian School of Business,COURSE,4.8,Mixed,320k,320000


## Phase 2 Cleaning Summary

- Removed the unnamed old index column because it was not a real course feature.
- Renamed columns to clear, consistent names.
- Confirmed that the dataset has no missing values.
- Confirmed that the dataset has no duplicate full rows.
- Found repeated course titles that should be considered during analysis.
- Converted enrollment values from text into whole numbers.
- Confirmed that ratings are numeric and within the expected 0 to 5 range.
- Checked certificate type and difficulty category values.
- Saved the cleaned dataset to `../outputs/coursera_courses_cleaned.csv`.